In [1]:
!pip install -q \
transformers \
accelerate \
openpyxl \
scikit-learn
!pip install -q sentencepiece

In [2]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

In [3]:
from google.colab import files

uploaded = files.upload()

Saving sinhala_youtube_comments_for_annotation.xlsx to sinhala_youtube_comments_for_annotation.xlsx


In [4]:
df = pd.read_excel(
    "/content/sinhala_youtube_comments_for_annotation.xlsx"
)

In [5]:
print(df.columns)

Index(['id', 'comment', 'label'], dtype='object')


In [6]:
df = df[['comment', 'label']]

In [7]:
df.dropna(inplace=True)

In [8]:
df['label'] = df['label'].astype(int)

In [9]:
print(df.head())

print(df['label'].value_counts())

                                             comment  label
0                                           පියතුමනි      0
1                                       නියම යි❤❤❤❤❤      0
2                              හිච්චගේ මුණ තමා maru😂      0
3  සානක අයියගෙ වීඩියෝ බලනවා නම් හිනා වෙන්න පුලුවන...      1
4  අයියා වෙඩින් මුද්ද නම් දාගෙනම නේද ටික් ටොක් කර...      0
label
0    1577
1    1471
Name: count, dtype: int64


In [10]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['comment'].astype(str).tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

In [11]:
MODEL_NAME = "ai4bharat/IndicBERTv2-MLM-only"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

config.json:   0%|          | 0.00/639 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.75M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: ai4bharat/IndicBERTv2-MLM-only
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those p

In [12]:
train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=256
)

test_encodings = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=256
)

In [13]:
class SarcasmDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):

        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item['labels'] = torch.tensor(
            self.labels[idx]
        )

        return item

    def __len__(self):

        return len(self.labels)

In [14]:
train_dataset = SarcasmDataset(
    train_encodings,
    train_labels
)

test_dataset = SarcasmDataset(
    test_encodings,
    test_labels
)

In [15]:
def compute_metrics(pred):

    labels = pred.label_ids

    preds = pred.predictions.argmax(-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average='macro'
    )

    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [16]:
training_args = TrainingArguments(

    output_dir="./results",

    num_train_epochs=5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    warmup_steps=100,

    weight_decay=0.01,

    logging_dir="./logs",

    logging_steps=10,

    save_total_limit=1
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [17]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    compute_metrics=compute_metrics
)

In [18]:
trainer.train()

Step,Training Loss
10,0.695282
20,0.694243
30,0.678316
40,0.682218
50,0.710443
60,0.694748
70,0.717149
80,0.698853
90,0.756092
100,0.743339


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1525, training_loss=0.7000233421951044, metrics={'train_runtime': 783.8755, 'train_samples_per_second': 15.551, 'train_steps_per_second': 1.945, 'total_flos': 1603661882419200.0, 'train_loss': 0.7000233421951044, 'epoch': 5.0})

In [19]:
results = trainer.evaluate()

print(results)

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Training Loss,Validation Loss,Step,Accuracy,F1,Precision,Recall
0.698118,0.692268,1525,0.518033,0.341253,0.259016,0.500000


{'eval_loss': 0.6922683119773865, 'eval_accuracy': 0.5180327868852459, 'eval_f1': 0.3412526997840173, 'eval_precision': 0.25901639344262295, 'eval_recall': 0.5}


In [20]:
predictions = trainer.predict(test_dataset)

preds = predictions.predictions.argmax(-1)

cm = confusion_matrix(test_labels, preds)

print(cm)

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


[[316   0]
 [294   0]]


In [21]:
print(
    classification_report(
        test_labels,
        preds
    )
)

              precision    recall  f1-score   support

           0       0.52      1.00      0.68       316
           1       0.00      0.00      0.00       294

    accuracy                           0.52       610
   macro avg       0.26      0.50      0.34       610
weighted avg       0.27      0.52      0.35       610



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [22]:
model.save_pretrained("./indicbert_model")
tokenizer.save_pretrained("./indicbert_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./indicbert_model/tokenizer_config.json', './indicbert_model/tokenizer.json')

In [23]:
!zip -r sarcasm_model.zip sarcasm_model

	zip warning: name not matched: sarcasm_model

zip error: Nothing to do! (try: zip -r sarcasm_model.zip . -i sarcasm_model)


In [25]:
text = "අද නම් හරිම ලස්සන වැඩක් කරලා"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True
)

inputs = {key: value.to(device) for key, value in inputs.items()}

outputs = model(**inputs)

prediction = torch.argmax(outputs.logits, dim=1)

print("Prediction:", prediction.item())

if prediction.item() == 1:
    print("Sarcastic")
else:
    print("Non-Sarcastic")

Prediction: 0
Non-Sarcastic
